# Week 6 — Frontend Architecture & UI Design
### Cardiovascular Disease Risk Prediction System (CardioML)

This notebook documents the frontend design, interface specification, and component architecture for the **CardioML** web interface in accordance with the Darshan University MLDL SOP Project guidelines (Pages 7–9).

#### Week 6 Objectives:
1. **User Interface Specifications:** Design a responsive, clinical-grade interface for patient risk assessment.
2. **Landing Page Architecture (SOP Page 7):** Hero banner, performance benchmarks, and core system capabilities.
3. **Interactive Prediction Form (SOP Page 8):** Multi-factor input form with dynamic client-side BMI and pulse pressure calculation.
4. **Model Transparency Dashboard (SOP Page 8):** Hyperparameters, validation metrics, and feature importance visualizer.
5. **Data Insights & Clinical Context (SOP Page 9):** Dataset summary, data cleaning statistics, understanding CVD, and healthy target ranges.

## 1. UI Design System & Component Hierarchy
The web application adopts a modern, clean healthcare design language with standard color tokens:
- **Primary Action Color:** `#ef4444` / `#f97316` (Medical Coral/Red for call-to-actions and alerts)
- **Accent / Health Color:** `#10b981` (Emerald Green for low-risk indicators and healthy ranges)
- **Neutral Backgrounds:** `#f8fafc` to `#ffffff` (Clean clinical whites and slates)
- **Typography:** Inter / system font family for clean readability across desktop and mobile devices.

In [1]:
# Define frontend navigation and page routes specification
routes_config = [
    {"name": "Home", "path": "/", "purpose": "Landing page with system highlights & accuracy metrics"},
    {"name": "Predict Now", "path": "/predict", "purpose": "Interactive risk prediction form with dynamic calculation"},
    {"name": "Model Info", "path": "/model", "purpose": "Model specifications, hyperparameters, and feature importance"},
    {"name": "Data Insights", "path": "/insights", "purpose": "EDA summary, dataset cleaning pipeline, and clinical target ranges"},
    {"name": "Disclaimer", "path": "/disclaimer", "purpose": "Academic and clinical guidance notice"}
]

import pandas as pd
pd.DataFrame(routes_config)

## 2. Dynamic Client-Side Feature Calculation
To enhance user experience and ensure consistency with the trained model's feature space, two key clinical features are calculated dynamically on the client before submission:
1. **Body Mass Index (BMI):**
   $$\text{BMI} = \frac{\text{Weight (kg)}}{(\text{Height (m)})^2} = \frac{\text{weight}}{(\text{height} / 100)^2}$$
2. **Pulse Pressure:**
   $$\text{Pulse Pressure} = \text{ap\_hi (Systolic)} - \text{ap\_lo (Diastolic)}$$

In [2]:
def calculate_derived_features(height_cm, weight_kg, ap_hi, ap_lo):
    """Calculates BMI and pulse pressure with clinical classification."""
    height_m = height_cm / 100.0
    bmi = round(weight_kg / (height_m ** 2), 2)
    pulse_pressure = int(ap_hi - ap_lo)
    
    # Clinical categorization
    if bmi < 18.5:
        bmi_cat = "Underweight"
    elif bmi < 25.0:
        bmi_cat = "Normal"
    elif bmi < 30.0:
        bmi_cat = "Overweight"
    else:
        bmi_cat = "Obese"
        
    pp_status = "Normal (30-50 mmHg)" if 30 <= pulse_pressure <= 50 else "Elevated / Abnormal"
    
    return {
        "bmi": bmi,
        "bmi_category": bmi_cat,
        "pulse_pressure": pulse_pressure,
        "pulse_pressure_status": pp_status
    }

# Example test with normal physiological values
sample_calc = calculate_derived_features(height_cm=175, weight_kg=70.5, ap_hi=120, ap_lo=80)
print("Sample Derived Features:", sample_calc)

## 3. Input Validation Bounds (Safety Guardrails)
The form enforces strict physiological boundaries to reject impossible data entry values, matching the cleaning filters applied in Week 2:

In [3]:
validation_rules = {
    "age_years": {"min": 18, "max": 100, "default": 50, "label": "Age (years)"},
    "height": {"min": 120, "max": 220, "default": 165, "label": "Height (cm)"},
    "weight": {"min": 30.0, "max": 200.0, "default": 70.0, "label": "Weight (kg)"},
    "ap_hi": {"min": 80, "max": 250, "default": 120, "label": "Systolic BP (ap_hi)"},
    "ap_lo": {"min": 40, "max": 180, "default": 80, "label": "Diastolic BP (ap_lo)"},
    "gender": {"options": [{"val": 1, "label": "Female"}, {"val": 2, "label": "Male"}]},
    "cholesterol": {"options": [{"val": 1, "label": "Normal"}, {"val": 2, "label": "Above Normal"}, {"val": 3, "label": "Well Above Normal"}]},
    "gluc": {"options": [{"val": 1, "label": "Normal"}, {"val": 2, "label": "Above Normal"}, {"val": 3, "label": "Well Above Normal"}]},
    "smoke": {"options": [{"val": 0, "label": "No"}, {"val": 1, "label": "Yes"}]},
    "alco": {"options": [{"val": 0, "label": "No"}, {"val": 1, "label": "Yes"}]},
    "active": {"options": [{"val": 1, "label": "Yes"}, {"val": 0, "label": "No"}]}
}

print(f"Total validated input parameters: {len(validation_rules)}")

## 4. Frontend Component Specifications (SOP Alignment)

### A. Landing Page (`templates/index.html` - SOP Page 7)
- **Navbar:** CardioML brand logo, links to *Data Insights*, *Model Info*, *Disclaimer*, and *Predict Now ↗* CTA.
- **Hero Section:** Badge `Early Warning System`, headline `Cardiovascular Risk Assessment`, subhead `Our AI model analyzes clinical variables to predict potential heart risks with precision.`
- **Four Performance Cards:**
  1. `73.4% Accuracy` (Trained on 70,000+ patient records with cross-validation)
  2. `Instant Results` (Real-time gradient boosting classification in under 5 seconds)
  3. `Secure & Private` (End-to-end encrypted processing with anonymous data)
  4. `Confidence Scores` (Predictions include probability-based confidence levels)

### B. Prediction Screen (`templates/predict.html` - SOP Page 8)
- Header with `< Back to Home` navigation and `Gradient Classifier` tag.
- Clean form card with two columns for personal and lifestyle variables.
- Interactive results display showing Risk Status (High vs. Low), Risk Probability percentage, probability meter, and personal risk factor alerts.

### C. Model Details View (`templates/model.html` - SOP Page 8 bottom)
- Four structured cards:
  1. **Model:** Algorithm name, framework (`scikit-learn`), feature count (13), trained date.
  2. **Hyperparameters:** Estimators (300), Learning Rate (0.05), Max Depth (4), Min Samples/Leaf (3).
  3. **Performance:** Accuracy (73.04%), F1 Score (71.53%), ROC-AUC (79.88%).
  4. **Feature Importance:** Visual progress bars for top predictors (`ap_hi`: 70.4%, `age_years`: 13.5%, `cholesterol`: 7.4%, `bmi`: 2.8%).

### D. Data Insights View (`templates/insights.html` - SOP Page 9)
- Three dataset cards: Raw records (70,000), Rows removed (1,427 / 2.04%), Final records (68,573).
- **Understanding CVD:** Clinical background, impact, prevention, and ML screening role.
- **Ideal Ranges:** Standard clinical targets for blood pressure, cholesterol, fasting glucose, BMI, and physical activity.

## Summary of Week 6 Deliverables
- Complete UI architecture drafted matching every component in the Darshan University MLDL SOP project guidelines.
- Dynamic client-side calculation logic specified for instant feedback on derived metrics.
- Layout and styling templates prepared for integration with the Flask backend in Week 7.